# MESSAGEix-Pakistan 
## Run script for MESSAGE Pakistan model
   In this notebook, we are reading data and building baseline scenerios. Data was prepared in excel sheet, which we simply load and solve our model. This script also generate some plots and resultant data.

In [ ]:
# Import required Libraries 
import pandas as pd
import ixmp
import message_ix
from message_ix import make_df
import os

# for modefiles execuation
import matplotlib.pyplot as plt
import pyam
import pandas as pd 
from ixmp import Platform
from ixmp.reporting import configure
from message_ix.reporting import Reporter
from pathlib import Path

# Import required files
from modelFiles.plotter_pakistan import plot, plotter

# Report Libraries
from message_ix.reporting import Reporter
#from modelFiles.update_units import unit_correction
from modelFiles.update_units_pakistan import unit_correction
import numpy as np

%matplotlib inline

In [ ]:
# Loading Modeling platform
mp = ixmp.Platform()

In [ ]:
# Creating a new, empty scenario
scenario = message_ix.Scenario(
    mp, model="Pakistan Model", scenario="baseline_xlsx", version="new"
)

While loading data, It's very important to pass aditional parameters like "init_items=True" in order to initialize items in a scenario with default values otherwise it will give some error.

In [ ]:
# Add your project root folder path
root = os.chdir(r"C:\Users\umer.yasin\Desktop\NEST_Pakistan")
path = "data\data_MESSAGE_PK.xlsx"

# Load data into our model - Current latest Data file is 
scenario.read_excel(path, add_units=True, commit_steps=False, init_items=True)

#### Correct Units
We use unit_correction function from modelFiles/update_units

In [ ]:
unit_correct = True  # unit s must be corrected to run this script
from_excel = False
min_year = 2015
max_year = 2060

if unit_correct:
    unit_correction(mp, scenario)
    # caseName = scenario.model + '__' + scenario.scenario + '__v' + str(scenario.version)
    # Solving the model
    #scenario.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

In [ ]:
# # Save unit-corrected scenario to new excel sheet
# scenario.to_excel('data/MESSAGE_PAK__test__v5.xlsx')

In [ ]:
# In order to commit or solve this model, we need to check out a scenario from the scenario database for editing.
scenario.check_out()

In [ ]:
# Commit changes into the database checked
scenario.commit(comment="Add all data from excel file to scenario")
scenario.set_as_default()

In [ ]:
# Exporting the built model (Scenario) to GAMS with an optional case name
caseName = scenario.model + '__' + scenario.scenario + '__v' + str(scenario.version)
# Solve model
scenario.solve()

In [ ]:
# In order to retrieve the value of a variable in a scenario. 
# In this case, the argument "OBJ" is passed to the method, which refers to the objective function variable. 
# The square brackets and "lvl" are used to retrieve the level of the variable, which represents its current value.
scenario.var("OBJ")["lvl"]

## Postprocessing and ploting results

In [ ]:
# Define
path = r"output"

# Plot and save results
# 1) First run plotter function
alldf = plotter(scenario, caseName, path)

# 2) Then Run Plot function
plot(alldf, caseName, path)

### Report 2

Make Areal graphs for (Demand by sector, CO2 Emissions, Primary energy supply by fuel, and total operation cost) on model output data

1) Plot for Primary Energy by fuel - Baseline Scenario

2) Plot for demands by sector

3. Plot for CO2 Emissions - Baseline Scenario

4. Plot for Total (Operational + Investment) Costs - Baseline

In [ ]:
from modelFiles.reporter import get_report_df, primary_energy_by_fuel_plot, demand_by_sector_plot, emission_plots, operation_investment_cost_plot

df = get_report_df(scenario)

primary_energy_by_fuel_plot(df)
demand_by_sector_plot(df)
emission_plots(df)
operation_investment_cost_plot(df)

In [ ]:
# closed the database connection, you can reopen it using "mp.open_db()"
mp.close_db()